## Import 

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from tensorflow import keras

/Users/fadykeliny/Desktop/lilaaa/jigsaw-toxic-comment-classification-challenge/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.15.0
GPU available: True


## CSV

In [4]:
LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
NUM_LABELS = len(LABEL_COLS)

df = pd.read_csv(os.path.join("train.csv"))

print(f"Total examples : {len(df):,}")
print(f"\nLabel distribution (% positive):")
print((df[LABEL_COLS].mean() * 100).round(2).to_string())
print()
print(df.head(3))

Total examples : 159,571

Label distribution (% positive):
toxic            9.58
severe_toxic     1.00
obscene          5.29
threat           0.30
insult           4.94
identity_hate    0.88

                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  


In [5]:
train_df, temp_df = train_test_split(df,      test_size=0.2,  random_state=SEED, stratify=df["toxic"])
val_df,   test_df = train_test_split(temp_df, test_size=0.5,  random_state=SEED, stratify=temp_df["toxic"])

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

Train : 127,656
Val   : 15,957
Test  : 15,958


## Clean the Text 

In [6]:
def preprocess_text(text: str) -> str:
    """Lowercase, strip non-letter characters, collapse whitespace."""
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_texts = [preprocess_text(t) for t in train_df["comment_text"]]
val_texts   = [preprocess_text(t) for t in val_df["comment_text"]]
test_texts  = [preprocess_text(t) for t in test_df["comment_text"]]

train_labels = train_df[LABEL_COLS].values.astype(np.float32)
val_labels   = val_df[LABEL_COLS].values.astype(np.float32)
test_labels  = test_df[LABEL_COLS].values.astype(np.float32)

print("Raw  :", train_df.iloc[0]["comment_text"][:80])
print("Clean:", train_texts[0][:80])
print("Labels:", train_labels[0])

Raw  : And more unfounded personal attacks by  here at Beeblebroxe's talk page. It just
Clean: nd more unfounded personal attacks by here at eeblebroxes talk page t just gets 
Labels: [0. 0. 0. 0. 0. 0.]


## Tokenization

In [7]:
MAX_VOCAB = 50000   
OOV_TOKEN = "<OOV>"  # out-of-vocabulary token (replaces UNK from the PyTorch version)

tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB,
    oov_token=OOV_TOKEN,
)
tokenizer.fit_on_texts(train_texts)   # build vocab from train only

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)  # +1 for the padding index 0

print(f"Vocabulary size (capped at {MAX_VOCAB:,}): {vocab_size:,}")
print("Example entries:", list(tokenizer.word_index.items())[:10])

Vocabulary size (capped at 50,000): 50,000
Example entries: [('<OOV>', 1), ('the', 2), ('to', 3), ('of', 4), ('and', 5), ('a', 6), ('you', 7), ('is', 8), ('that', 9), ('in', 10)]


In [8]:
# Encode to integer sequences
train_seqs = tokenizer.texts_to_sequences(train_texts)
val_seqs   = tokenizer.texts_to_sequences(val_texts)
test_seqs  = tokenizer.texts_to_sequences(test_texts)

# Determine MAX_LENGTH from training set (95th percentile)
seq_lengths = [len(s) for s in train_seqs]
MAX_LENGTH  = int(np.percentile(seq_lengths, 95))
print(f"95th-percentile sequence length -> MAX_LENGTH = {MAX_LENGTH}")

# Pad / truncate
train_padded = keras.preprocessing.sequence.pad_sequences(train_seqs, maxlen=MAX_LENGTH, padding="post", truncating="post")
val_padded   = keras.preprocessing.sequence.pad_sequences(val_seqs,   maxlen=MAX_LENGTH, padding="post", truncating="post")
test_padded  = keras.preprocessing.sequence.pad_sequences(test_seqs,  maxlen=MAX_LENGTH, padding="post", truncating="post")

print(f"train_padded shape : {train_padded.shape}")
print(f"val_padded shape   : {val_padded.shape}")
print(f"test_padded shape  : {test_padded.shape}")

95th-percentile sequence length -> MAX_LENGTH = 215
train_padded shape : (127656, 215)
val_padded shape   : (15957, 215)
test_padded shape  : (15958, 215)


In [9]:
BATCH_SIZE = 64

def make_dataset(sequences, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((sequences, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_padded, train_labels, shuffle=True)
val_ds   = make_dataset(val_padded,   val_labels)
test_ds  = make_dataset(test_padded,  test_labels)

# Quick sanity check
for x_batch, y_batch in train_ds.take(1):
    print(f"Batch x shape : {x_batch.shape}")  # (64, MAX_LENGTH)
    print(f"Batch y shape : {y_batch.shape}")  # (64, 6)


2026-06-13 20:33:22.302558: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-06-13 20:33:22.302917: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-06-13 20:33:22.302970: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-06-13 20:33:22.303341: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-13 20:33:22.303822: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Batch x shape : (64, 215)
Batch y shape : (64, 6)


In [10]:
EMBEDDING_DIM = 128
HIDDEN_DIM    = 128

def build_lstm_model(vocab_size, embedding_dim, hidden_dim, num_labels, max_length):
    model = keras.Sequential([
        # 1. Embedding: mask_zero=True tells downstream layers to ignore padding tokens
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_length,
            mask_zero=True,
            name="embedding"
        ),

        # 2. SpatialDropout1D regularises at the feature-map level
        layers.SpatialDropout1D(0.3, name="spatial_dropout"),

        # 3. Bidirectional LSTM — reads forward and backward, concatenates both final states
        #    return_sequences=False → only the final hidden state (many-to-one)
        layers.Bidirectional(
            layers.LSTM(hidden_dim, dropout=0.2, recurrent_dropout=0.1),
            name="bilstm"
        ),

        # 4. Dense projection
        layers.Dense(64, activation="relu", name="dense_hidden"),

        # 5. Dropout
        layers.Dropout(0.5, name="dropout"),

        # 6. Output — sigmoid per label for multi-label classification
        layers.Dense(num_labels, activation="sigmoid", name="output"),
    ], name="lstm_toxic_classifier")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",   # each label treated independently
        metrics=[
            "binary_accuracy",
            keras.metrics.AUC(name="auc", multi_label=True),
        ],
    )
    return model


model = build_lstm_model(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LABELS, MAX_LENGTH)
model.summary()

Model: "lstm_toxic_classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 215, 128)          6400000   
                                                                 
 spatial_dropout (SpatialDr  (None, 215, 128)          0         
 opout1D)                                                        
                                                                 
 bilstm (Bidirectional)      (None, 256)               263168    
                                                                 
 dense_hidden (Dense)        (None, 64)                16448     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 output (Dense)              (None, 6)                 390       
                                             

In [ ]:
# 1. Re-compile right here with the M1-safe legacy optimizer to fix the crash
model.compile(
    optimizer=keras.optimizers.legacy.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "binary_accuracy",
        keras.metrics.AUC(name="auc", multi_label=True),
    ],
)

# 2. Your training setup
NUM_EPOCHS = 15

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        patience=3,
        mode="max",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

# 3. Start training (keeping your verbose=1 so you can watch the standard progress bar)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/15


2026-06-13 20:33:25.070226: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


   2/1995 [..............................] - ETA: 144:54:31 - loss: 0.6896 - binary_accuracy: 0.5911 - auc: 0.5074

## Learning Curves

In [ ]:
hist       = history.history
epochs_run = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_run, hist["loss"],     marker="o", label="Train loss")
axes[0].plot(epochs_run, hist["val_loss"], marker="o", label="Val loss")
axes[0].set_title("Binary Cross-Entropy Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_run, hist["binary_accuracy"],     marker="o", label="Train acc")
axes[1].plot(epochs_run, hist["val_binary_accuracy"], marker="o", label="Val acc")
axes[1].set_title("Binary Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

# AUC
axes[2].plot(epochs_run, hist["auc"],     marker="o", label="Train AUC")
axes[2].plot(epochs_run, hist["val_auc"], marker="o", label="Val AUC")
axes[2].set_title("Multi-label ROC-AUC")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation on the Test Set

In [ ]:
test_results = model.evaluate(test_ds, verbose=0)
print("Test set results:")
for name, val in zip(model.metrics_names, test_results):
    print(f"  {name:25s}: {val:.4f}")

# Per-label AUC with sklearn (more granular than Keras's aggregate metric)
y_pred = model.predict(test_ds, verbose=0)

print("\nPer-label ROC-AUC:")
for i, label in enumerate(LABEL_COLS):
    auc = roc_auc_score(test_labels[:, i], y_pred[:, i])
    print(f"  {label:20s}: {auc:.4f}")

mean_auc = roc_auc_score(test_labels, y_pred, average="macro")
print(f"\n  Mean ROC-AUC (macro): {mean_auc:.4f}")

##  Inference on New Comments

In [ ]:
def predict_toxicity(texts, threshold=0.5):
    """Return probabilities and binary predictions for a list of raw strings."""
    cleaned = [preprocess_text(t) for t in texts]
    seqs    = tokenizer.texts_to_sequences(cleaned)
    padded  = keras.preprocessing.sequence.pad_sequences(
                  seqs, maxlen=MAX_LENGTH, padding="post", truncating="post")
    probs   = model.predict(padded, verbose=0)
    preds   = (probs >= threshold).astype(int)

    prob_df = pd.DataFrame(probs, columns=LABEL_COLS)
    pred_df = pd.DataFrame(preds, columns=LABEL_COLS)
    return prob_df, pred_df


sample_comments = [
    "You are an absolute idiot and I hope you disappear forever.",
    "Great point! I completely agree with your analysis.",
    "This is the worst article I have ever read, pure garbage.",
]

probs, preds = predict_toxicity(sample_comments)

print("Predicted probabilities:")
print(probs.round(3).to_string())
print("\nBinary predictions (threshold = 0.5):")
print(preds.to_string())